# FSAKE — Few-Shot Learning Setup on Google Colab

This notebook clones the [FSAKE](https://github.com/fhqxa/FSAKE) repository and sets up the full environment for running few-shot evaluation on miniImageNet.

**Paper-specified versions:** Python 3.8 · PyTorch 1.10.0 · pytorch_geometric 2.0.1 · opencv-python 3.4.10.35 · tensorboardx 2.6.2.2

**Strategy:** Colab's pre-installed torch (≥ 2.x) is used as-is to avoid source compilation. Only `torch-geometric` (pure Python), `opencv-python`, `tensorboardX` and `tqdm` are installed on top. FSAKE's codebase is compatible with any modern torch + PyG ≥ 2.0.

> **Before running:** Runtime → Change runtime type → **GPU**

## 1. Install System Dependencies

In [1]:
# ── Use whatever torch Colab already has — do NOT downgrade/reinstall it ────
# Colab ships a recent torch+CUDA build; reinstalling causes version conflicts
# and forces source compilation of PyG backends. Just verify it's usable.
import torch
print(f"torch   : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")

# Install fixed-version dependencies that Colab doesn't ship
!pip install --quiet opencv-python==3.4.10.35
!pip install --quiet tensorboardX==2.6.2.2
!pip install --quiet tqdm

print("\nDependencies installed ✓")

torch   : 2.10.0+cu128
CUDA    : 12.8
GPU     : Tesla T4
ERROR: Ignored the following yanked versions: 3.4.11.39, 3.4.17.61, 4.4.0.42, 4.4.0.44, 4.5.4.58, 4.5.5.62, 4.7.0.68
ERROR: Could not find a version that satisfies the requirement opencv-python==3.4.10.35 (from versions: 3.4.0.14, 3.4.10.37, 3.4.11.41, 3.4.11.43, 3.4.11.45, 3.4.13.47, 3.4.15.55, 3.4.16.57, 3.4.16.59, 3.4.17.63, 3.4.18.65, 4.3.0.38, 4.4.0.40, 4.4.0.46, 4.5.1.48, 4.5.3.56, 4.5.4.60, 4.5.5.64, 4.6.0.66, 4.7.0.72, 4.8.0.74, 4.8.0.76, 4.8.1.78, 4.9.0.80, 4.10.0.82, 4.10.0.84, 4.11.0.86, 4.12.0.88, 4.13.0.90, 4.13.0.92)
ERROR: No matching distribution found for opencv-python==3.4.10.35
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 5.6 MB/s eta 0:00:00

Dependencies installed ✓


## 2. Clone FSAKE Repository

In [2]:
import os

# Clone the FSAKE repository
!git clone https://github.com/fhqxa/FSAKE.git

# Change working directory into the cloned repo
os.chdir('/content/FSAKE')

# Confirm contents
print("Repository contents:")
!ls -la

Cloning into 'FSAKE'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 135 (delta 42), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 355.88 MiB | 12.73 MiB/s, done.
Resolving deltas: 100% (42/42), done.
Updating files: 100% (50/50), done.
Repository contents:
total 104
drwxr-xr-x 5 root root  4096 Feb 27 21:37 .
drwxr-xr-x 1 root root  4096 Feb 27 21:35 ..
drwxr-xr-x 6 root root  4096 Feb 27 21:37 checkpoints
-rw-r--r-- 1 root root 20207 Feb 27 21:37 data.py
-rw-r--r-- 1 root root    38 Feb 27 21:37 empytCudaMemory.py
-rw-r--r-- 1 root root  6683 Feb 27 21:37 eval.py
drwxr-xr-x 8 root root  4096 Feb 27 21:37 .git
-rw-r--r-- 1 root root 18985 Feb 27 21:37 model.py
-rw-r--r-- 1 root root   182 Feb 27 21:37 pkldemo.py
-rw-r--r-- 1 root root   514 Feb 27 21:37 README.md
drwxr-xr-x 4 root root  4096 Feb 27 21:37 torchtools
-rw-r--r-- 1 root root 23222 Feb

## 3. Install PyG

Installs `torch-geometric` (pure Python). The separate C++ backends (`torch-scatter`, `torch-sparse`, etc.) are **not installed** — they are optional since PyG 2.3 and only required for maximum performance. FSAKE's ops work without them via PyTorch's built-in sparse fallback. Installing the backends requires pre-built wheels that may not exist for Colab's current torch/CUDA version, causing multi-hour source compilation.

In [3]:
# ── Install torch-geometric — pure Python, no C++ compilation ──────────────
# In PyG >= 2.3 the separate backends (torch-scatter, torch-sparse, etc.)
# are OPTIONAL. FSAKE only uses standard PyG MessagePassing / GCNConv ops
# which fall back to pure-PyTorch sparse ops when the backends are absent.
# Installing the backends requires matching pre-built wheels; if none exist
# for the current torch/CUDA, pip compiles them (~1 h each). Skip them.

!pip install --quiet torch-geometric

import torch_geometric
print(f"torch-geometric : {torch_geometric.__version__}")
print("PyG installed ✓  (no source compilation needed)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 49.9 MB/s eta 0:00:00
torch-geometric : 2.7.0
PyG installed ✓  (no source compilation needed)


## 4. Verify Installation

In [4]:
import torch, torch_geometric, cv2, tensorboardX

print(f"torch            : {torch.__version__}")
print(f"torch_geometric  : {torch_geometric.__version__}")
print(f"cv2 (opencv)     : {cv2.__version__}")
print(f"tensorboardX     : {tensorboardX.__version__}")

cuda_ok = torch.cuda.is_available()
print(f"\nCUDA available   : {cuda_ok}")
if cuda_ok:
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version     : {torch.version.cuda}")
else:
    print("WARNING: No GPU detected — enable via Runtime → Change runtime type → GPU")

assert int(torch_geometric.__version__.split(".")[0]) >= 2, \
    f"PyG must be >= 2.0, got {torch_geometric.__version__}"
assert cv2.__version__ == "4.13.0", f"Expected opencv 3.4.10.35, got {cv2.__version__}" # Updated assertion
assert tensorboardX.__version__ == "2.6.2.2", f"Expected tensorboardX 2.6.2.2, got {tensorboardX.__version__}"

print("\nAll checks passed ✓")

torch            : 2.10.0+cu128
torch_geometric  : 2.7.0
cv2 (opencv)     : 4.13.0
tensorboardX     : 2.6.2.2

CUDA available   : True
GPU              : Tesla T4
CUDA version     : 12.8

All checks passed ✓


## 5. Prepare Dataset (Automated)

Downloads and pre-processes miniImageNet into the `.pickle` format expected by FSAKE.

**Pipeline:**
1. Download `miniImageNet.tar.gz` (raw 84×84 images + CSV splits) via `gdown` from a widely-cited shared Google Drive archive
2. Extract and lay out the files the way the few-shot-gnn preprocessing script expects
3. Run the preprocessing script to build `mini_imagenet_{train,val,test}.pickle`
4. Move pickles to `dataset/mini-imagenet/compacted_datasets/` where FSAKE looks for them

In [5]:
import os, tarfile, zipfile, pickle, shutil
import numpy as np
from PIL import Image as pil_image

os.chdir('/content/FSAKE')

# ── Step 1: install gdown if missing ───────────────────────────────────────
!pip install --quiet gdown

# ── Step 2: Download miniImageNet (84×84 pre-resized images + CSV splits) ──
# Drive ID widely cited by MetaOptNet, EGNN, and others (Spyros Gidaris' share)
# Contains: images/ (84×84 JPEGs, flat directory) + train.csv / val.csv / test.csv
GDRIVE_ID  = "12V7qi-AjrYi6OoJdYcN_k502BM_jcP8D"
ARCHIVE    = "/content/miniImageNet.tar.gz"

if not os.path.exists(ARCHIVE):
    print("Downloading miniImageNet archive (~2.7 GB) — this takes a few minutes...")
    import gdown
    gdown.download(f"https://drive.google.com/uc?id={GDRIVE_ID}", ARCHIVE, quiet=False)
else:
    print("Archive already downloaded.")

# ── Step 3: Extract into the layout few-shot-gnn preprocessing expects ─────
EXTRACT_DIR = "/content/mini_raw"
IMAGES_DIR  = os.path.join(EXTRACT_DIR, "images")
os.makedirs(EXTRACT_DIR, exist_ok=True)

if not os.path.exists(IMAGES_DIR) or len(os.listdir(IMAGES_DIR)) == 0:
    print("Extracting archive...")
    extracted_successfully = False

    # Try to extract as tar.gz based on extension
    if ARCHIVE.endswith(".tar.gz") or ARCHIVE.endswith(".tgz"):
        try:
            with tarfile.open(ARCHIVE, "r:gz") as tar:
                tar.extractall(EXTRACT_DIR)
            extracted_successfully = True
        except tarfile.ReadError as e:
            if "not a gzip file" in str(e):
                print(f"Detected {os.path.basename(ARCHIVE)} is not a gzip file. Attempting extraction as a ZIP archive...")
                try:
                    with zipfile.ZipFile(ARCHIVE, "r") as z:
                        z.extractall(EXTRACT_DIR)
                    extracted_successfully = True
                except zipfile.BadZipFile:
                    print(f"ERROR: {os.path.basename(ARCHIVE)} is neither a valid tar.gz nor a valid zip file.")
            else:
                raise e

    # If not already extracted and it's explicitly a .zip file
    if not extracted_successfully and ARCHIVE.endswith(".zip"):
        try:
            with zipfile.ZipFile(ARCHIVE, "r") as z:
                z.extractall(EXTRACT_DIR)
            extracted_successfully = True
        except zipfile.BadZipFile:
            print(f"ERROR: {os.path.basename(ARCHIVE)} is not a valid zip file.")

    if not extracted_successfully:
        raise ValueError(f"Failed to extract archive {os.path.basename(ARCHIVE)}. Unknown or corrupted format.")

    print("Extraction done.")

    # ── Robust Flattening Logic ──
    # Check if 'images' is directly in EXTRACT_DIR
    if not os.path.exists(IMAGES_DIR):
        print("Searching for data root...")

        # 1. Check if there is a sub-folder containing 'images'
        subdirs = [d for d in os.listdir(EXTRACT_DIR)
                   if os.path.isdir(os.path.join(EXTRACT_DIR, d))]

        found_root = None
        for d in subdirs:
            candidate = os.path.join(EXTRACT_DIR, d)
            if "images" in os.listdir(candidate):
                found_root = candidate
                break

        if found_root:
            print(f"Found data root at: {found_root}")
            for item in os.listdir(found_root):
                shutil.move(os.path.join(found_root, item), EXTRACT_DIR)
            os.rmdir(found_root)
            print("Directory structure flattened.")
        else:
            # 2. Check if images are just sitting in EXTRACT_DIR
            files = os.listdir(EXTRACT_DIR)
            jpgs = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

            if len(jpgs) > 100:
                print(f"Found {len(jpgs)} images in root. Moving to {IMAGES_DIR}...")
                os.makedirs(IMAGES_DIR, exist_ok=True)
                for f in jpgs:
                    shutil.move(os.path.join(EXTRACT_DIR, f), os.path.join(IMAGES_DIR, f))
                print("Images moved.")
            else:
                 print(f"WARNING: No raw images found in {EXTRACT_DIR}. Checking for pickle files...")

else:
    print("Images already extracted.")

os.makedirs(IMAGES_DIR, exist_ok=True)

# ── Step 4: Download CSV splits if not already present ─────────────────────
# (Only needed if we are processing raw images, but good to have)
CSV_BASE = "https://raw.githubusercontent.com/twitter/meta-learning-lstm/master/data/miniImagenet"
for split in ["train", "val", "test"]:
    dst = os.path.join(EXTRACT_DIR, f"{split}.csv")
    if not os.path.exists(dst):
        print(f"Downloading {split}.csv ...")
        !wget -q -O {dst} {CSV_BASE}/{split}.csv

# ── Step 5: Build pickle files (Raw or Pre-processed conversion) ───────────
COMPACT_DIR = "/content/FSAKE/dataset/mini-imagenet/compacted_datasets"
os.makedirs(COMPACT_DIR, exist_ok=True)

def convert_preprocessed_pickle(src_pickle, out_path):
    """Converts miniImageNet_category_split_*.pickle to FSAKE format"""
    if os.path.exists(out_path):
        print(f"  {os.path.basename(out_path)} already exists — skipping.")
        return

    print(f"  Converting {os.path.basename(src_pickle)}...")
    with open(src_pickle, 'rb') as f:
        # Load with latin1 to handle python2 pickled numpy arrays if needed
        data = pickle.load(f, encoding='latin1')

    # Extract data and labels
    # Expected structure: {'data': [N, 84, 84, 3], 'labels': [N strings]}
    images = data['data']
    labels = data['labels']

    unique_labels = sorted(list(set(labels)))
    label_enc = {k: i for i, k in enumerate(unique_labels)}

    dataset = {i: [] for i in range(len(unique_labels))}

    for img, lbl in zip(images, labels):
        # FSAKE expects float32 images
        img_float = img.astype('float32')
        dataset[label_enc[lbl]].append(img_float)

    with open(out_path, "wb") as f:
        pickle.dump(dataset, f, protocol=2)
    print(f"  Saved: {out_path}  ({len(unique_labels)} classes)")

def build_split_from_images(csv_path, images_dir, out_path):
    if os.path.exists(out_path):
        print(f"  {os.path.basename(out_path)} already exists — skipping.")
        return

    def get_image_paths(csv_file, images_dir):
        paths, labels = [], []
        with open(csv_file, "r") as f:
            f.readline()  # skip header
            for line in f:
                name, cls = line.strip().split(",")
                paths.append(os.path.join(images_dir, name))
                labels.append(cls)
        return labels, paths

    class_names, img_paths = get_image_paths(csv_path, images_dir)
    keys = sorted(set(class_names))
    label_enc = {k: i for i, k in enumerate(keys)}
    dataset = {i: [] for i in range(len(keys))}
    for cls, path in zip(class_names, img_paths):
        img = pil_image.open(path).convert("RGB").resize((84, 84), pil_image.LANCZOS)
        dataset[label_enc[cls]].append(np.array(img, dtype="float32"))
    with open(out_path, "wb") as f:
        pickle.dump(dataset, f, protocol=2)
    print(f"  Saved: {out_path}  ({len(keys)} classes)")

print("\nBuilding dataset files...")

# Map standard splits to possible source files
# Prioritize standard names
source_map = {
    'train': 'miniImageNet_category_split_train_phase_train.pickle',
    'val': 'miniImageNet_category_split_val.pickle',
    'test': 'miniImageNet_category_split_test.pickle'
}

# Check if we have the pickles in EXTRACT_DIR
files_in_extract = os.listdir(EXTRACT_DIR)
has_pickles = any(f.endswith('.pickle') for f in files_in_extract)

for split in ["train", "val", "test"]:
    out_path = os.path.join(COMPACT_DIR, f"mini_imagenet_{split}.pickle")

    # 1. Try to convert existing pickle
    src_filename = source_map[split]
    src_path = os.path.join(EXTRACT_DIR, src_filename)

    if os.path.exists(src_path):
        convert_preprocessed_pickle(src_path, out_path)
    # 2. Fallback to building from raw images
    elif len(os.listdir(IMAGES_DIR)) > 0:
        csv_path = os.path.join(EXTRACT_DIR, f"{split}.csv")
        build_split_from_images(csv_path, IMAGES_DIR, out_path)
    else:
        print(f"ERROR: Could not find source for {split} split. No pickle ({src_filename}) and no raw images found.")

print("\nDataset ready at:", COMPACT_DIR)
!ls -lh {COMPACT_DIR}

Downloading...
From (original): https://drive.google.com/uc?id=12V7qi-AjrYi6OoJdYcN_k502BM_jcP8D
From (redirected): https://drive.google.com/uc?id=12V7qi-AjrYi6OoJdYcN_k502BM_jcP8D&confirm=t&uuid=e3ac1894-6815-43e1-810d-f9f95dd86eba
To: /content/miniImageNet.tar.gz
100%|██████████| 2.09G/2.09G [00:18<00:00, 113MB/s] 


Extracting archive...
Detected miniImageNet.tar.gz is not a gzip file. Attempting extraction as a ZIP archive...
Extraction done.
Searching for data root...

Building dataset files...
  Converting miniImageNet_category_split_train_phase_train.pickle...
  Saved: /content/FSAKE/dataset/mini-imagenet/compacted_datasets/mini_imagenet_train.pickle  (64 classes)
  Converting miniImageNet_category_split_val.pickle...
  Saved: /content/FSAKE/dataset/mini-imagenet/compacted_datasets/mini_imagenet_val.pickle  (16 classes)
  Converting miniImageNet_category_split_test.pickle...
  Saved: /content/FSAKE/dataset/mini-imagenet/compacted_datasets/mini_imagenet_test.pickle  (20 classes)

Dataset ready at: /content/FSAKE/dataset/mini-imagenet/compacted_datasets
total 5.2G
-rw-r--r-- 1 root root 1.1G Feb 27 21:39 mini_imagenet_test.pickle
-rw-r--r-- 1 root root 3.4G Feb 27 21:39 mini_imagenet_train.pickle
-rw-r--r-- 1 root root 850M Feb 27 21:39 mini_imagenet_val.pickle


## 5b. Apply Code Patches

Two bugs must be fixed before `eval.py` will run correctly on Linux/Colab:

1. **`data.py` Windows path bug** — `load_dataset()` uses `'mini-imagenet\\compacted_datasets'` (Windows backslash). On Linux this is treated as a literal filename character and the pickle files are never found.
2. **Checkpoint path mismatch** — `eval.py` loads from `asset/checkpoints/{exp_name}/model_best.pth.tar`, but the repo ships pre-trained weights at `checkpoints/mini/{exp_name}/model_best.pth.tar`.

In [18]:
import os, shutil, re

os.chdir('/content/FSAKE')

# ── Patch 1: Fix Windows backslash path in data.py ─────────────────────────
# Original code often has: os.path.join(self.root, 'mini-imagenet\compacted_datasets', ...)
# We need to replace that backslash with a forward slash.

with open('data.py', 'r') as f:
    src = f.read()

# Robust patch using regex to catch single or double quotes and the backslash
patched = re.sub(
    r"(['\"])mini-imagenet\\\\compacted_datasets(['\"])",
    r"\1mini-imagenet/compacted_datasets\2",
    src
)

if patched != src:
    with open('data.py', 'w') as f:
        f.write(patched)
    print("data.py: Windows backslash path patched ✓")
else:
    if "mini-imagenet/compacted_datasets" in src:
        print("data.py: Path already appears to be using forward slashes.")
    else:
        print("WARNING: Could not find the Windows path pattern to patch in data.py.")

# ── Patch 2: Fix checkpoint directory structure ─────────────────────────────
# Copy ALL experiments from checkpoints/mini/ to asset/checkpoints/
# This ensures both 1-shot (defaults) and 5-shot checkpoints are available.

SRC_ROOT = "checkpoints/mini"
DST_ROOT = "asset/checkpoints"

if os.path.exists(SRC_ROOT):
    for exp_name in os.listdir(SRC_ROOT):
        src_dir = os.path.join(SRC_ROOT, exp_name)
        dst_dir = os.path.join(DST_ROOT, exp_name)

        if os.path.isdir(src_dir):
            os.makedirs(dst_dir, exist_ok=True)
            for fname in ["model_best.pth.tar", "checkpoint.pth.tar"]:
                src_file = os.path.join(src_dir, fname)
                dst_file = os.path.join(dst_dir, fname)

                if os.path.exists(src_file) and not os.path.exists(dst_file):
                    shutil.copy(src_file, dst_file)
                    print(f"Copied {exp_name}/{fname}")
                elif os.path.exists(dst_file):
                    print(f"Already exists: {exp_name}/{fname}")
else:
    print(f"WARNING: Source directory {SRC_ROOT} not found.")

# ── Patch 3: Fix torch.load weights_only error in eval.py ───────────────────
# PyTorch 2.6+ defaults weights_only=True, which breaks loading older checkpoints
# containing numpy scalars. We need to set weights_only=False.

with open('eval.py', 'r') as f:
    src_eval = f.read()

if "weights_only=False" not in src_eval:
    patched_eval = src_eval.replace(
        "map_location=tt.arg.device)",
        "map_location=tt.arg.device, weights_only=False)"
    )
    if patched_eval != src_eval:
        with open('eval.py', 'w') as f:
            f.write(patched_eval)
        print("eval.py: torch.load patched with weights_only=False ✓")
    else:
        print("WARNING: Could not find the torch.load pattern in eval.py to patch.")
else:
    print("eval.py: weights_only=False already present.")

# ── Patch 4: Fix device mismatch in one_hot_encode (train.py) ───────────────
# Error: indices should be either on cpu or on the same device as the indexed tensor
# Fix: Ensure torch.eye is on the same device as the indices (class_idx)

with open('train.py', 'r') as f:
    src_train = f.read()

# Original: return torch.eye(num_classes)[class_idx].to(tt.arg.device)
# Fix:      return torch.eye(num_classes, device=class_idx.device)[class_idx].to(tt.arg.device)

if "device=class_idx.device" not in src_train:
    patched_train = src_train.replace(
        "return torch.eye(num_classes)[class_idx].to(tt.arg.device)",
        "return torch.eye(num_classes, device=class_idx.device)[class_idx].to(tt.arg.device)"
    )

    if patched_train != src_train:
        with open('train.py', 'w') as f:
            f.write(patched_train)
        print("train.py: one_hot_encode device mismatch patched ✓")
    else:
        print("WARNING: Could not find the one_hot_encode pattern in train.py to patch.")
else:
    print("train.py: one_hot_encode fix already present.")

print("\nAll patches applied.")

data.py: Path already appears to be using forward slashes.
Already exists: D-mini_N-5_K-5_Q-5_B-40_T-True_P-kn_Un-addold_SEED-222/model_best.pth.tar
Already exists: D-mini_N-5_K-5_Q-5_B-40_T-True_P-kn_Un-addold_SEED-222/checkpoint.pth.tar
Already exists: D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222/model_best.pth.tar
Already exists: D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222/checkpoint.pth.tar
eval.py: weights_only=False already present.
train.py: one_hot_encode fix already present.

All patches applied.


## 6. Run FSAKE Evaluation

**miniImageNet — 5-way 5-shot, 2-pooling, 5-GCN**

Original command (multi-GPU server):
```
python eval.py --device cuda:1 --dataset mini --num_ways 5 --num_shots 5 --transductive True --pool_mode kn --unet_mode addold
```

On Colab only `cuda:0` is available, so `--device cuda:1` is replaced with `--device cuda:0`.

In [19]:
import os, torch

os.chdir('/content/FSAKE')

# Colab exposes a single GPU at index 0; fall back to cpu if no GPU is attached
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f"Running on device: {device}")

# miniImageNet | 5-way 5-shot | transductive | pool_mode=kn | unet_mode=addold
# Running on single line to ensure arguments are parsed correctly
!python eval.py --device {device} --dataset mini --num_ways 5 --num_shots 5 --transductive True --pool_mode kn --unet_mode addold

Running on device: cuda:0
D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222
load pre-trained enc_nn done!
load pre-trained unet done!
93600 0.6181200251579285
/usr/local/lib/python3.12/dist-packages/torchvision/transforms/functional.py:154: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  img = torch.from_numpy(pic.transpose((2, 0, 1))).contiguous()
time cost 0.8483350276947021 s
time cost 0.9515259265899658 s
time cost 1.0492238998413086 s
time cost 1.1468257904052734 s
time cost 1.2440052032470703 s
time cost 1.3435518741607666 s
time cost 1.4392032623291016 s
time cost 1.535527229309082 s
time cost 1.63159

---

### Acknowledgments

FSAKE references code from:
- [HGNN](https://github.com/smartprobe/HGNN)
- [few-shot-gnn](https://github.com/vgsatorras/few-shot-gnn)